# 09 — Combined topology + prompt evolution

**Goal:** evolve **both** the topology and the prompts. But do it cheaply.

The naïve way would be to search the full space at once: every topology × every prompt combination. The cost grows as N_topology × N_prompts × N_eval. That's a four- or five-figure bill, fast.

There is a better way, used by [AFlow](https://arxiv.org/abs/2410.10762): **decouple** the two searches.

| Phase | What it changes | What stays fixed |
|---|---|---|
| **Phase 1** | The topology (notebook 08) | Strong baseline prompts |
| **Phase 2** | The prompts (notebook 06, GEPA) | The winning topology from phase 1 |

Two cheaper searches. The cost goes from N×M to N+M.

**This notebook walks through the full ablation** — four configurations:

| Configuration | Topology | Prompts |
|---|---|---|
| **Baseline** | seed | seed |
| **Topology only** | evolved | seed |
| **Prompts only** | seed | GEPA-evolved |
| **Combined** | evolved | GEPA-evolved on winner |

We measure each one. **Cost: ~$3–5.**


## 1. Setup

Define the agent class and the seed prompts. We use the same `CotAgent` for both `cot` and `verifier` roles — they differ only in their system prompt. That makes Phase 2 cleaner: GEPA evolves a per-role prompt, and we plug it in.


In [ ]:
from __future__ import annotations
from typing import Any
from pydantic import BaseModel, Field
from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()


class Answer(BaseModel):
    answer: str = Field(description="Direct answer or 'the paragraph does not say'.")


class CotAgent(BaseAgent[GlobalState, Answer]):
    async def _run_implementation(self, state, **kw):
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output


# Seed prompts — Phase 2 will mutate these.
SEED_COT_PROMPT = (
    "You answer questions concisely from the provided paragraph. "
    "If the paragraph does not contain the answer, reply 'the paragraph does not say'."
)
SEED_VERIFIER_PROMPT = (
    "You receive a question and a candidate answer. Verify the answer is correct given the paragraph. "
    "If incorrect, replace it with the correct answer. If the paragraph does not contain the answer, "
    "reply 'the paragraph does not say'."
)


def make_cot_with_prompt(prompt: str):
    def _f():
        return CotAgent(
            agent_name="cot",
            system_prompt=prompt,
            output_type=Answer,
            model=config.llm_model,
            api_key=config.llm_api_key,
        )
    return _f


def make_verifier_with_prompt(prompt: str):
    def _f():
        return CotAgent(
            agent_name="verifier",
            system_prompt=prompt,
            output_type=Answer,
            model=config.llm_model,
            api_key=config.llm_api_key,
        )
    return _f


def make_registry(cot_prompt: str, verifier_prompt: str):
    return {
        "cot": make_cot_with_prompt(cot_prompt),
        "verifier": make_verifier_with_prompt(verifier_prompt),
    }


seed_registry = make_registry(SEED_COT_PROMPT, SEED_VERIFIER_PROMPT)
print("agents:", sorted(seed_registry))

## 2. Gold set

Same three-bucket shape as notebook 08, halved to keep cost down (6 examples instead of 9).


In [ ]:
from orqest.optimization import GoldExample


def ex(question: str, paragraph: str, expected: str, bucket: str) -> GoldExample[str, Answer]:
    return GoldExample[str, Answer](
        input=f"Paragraph: {paragraph}\n\nQuestion: {question}",
        expected=Answer(answer=expected),
        id=bucket,
    )


gold_set = [
    ex("In what year was Python first released?",
       "Python was first released in 1991 by Guido van Rossum.", "1991", "factual"),
    ex("Who is the CEO of Tesla?",
       "Tesla, headquartered in Austin, is led by CEO Elon Musk.", "Elon Musk", "factual"),
    ex("How many years between Python and JavaScript releases?",
       "Python was released in 1991. JavaScript first appeared in 1995.", "4", "inferential"),
    ex("Which is older — the company or its CEO's tenure?",
       "Tesla was founded in 2003. Elon Musk became CEO in 2008.", "the company", "inferential"),
    ex("What is the speed of light in km/s?",
       "This paragraph discusses the history of jazz music in New Orleans during the 1920s.",
       "the paragraph does not say", "abstain"),
    ex("Who painted the Mona Lisa?",
       "Solar panels convert sunlight into electricity at 15-22% efficiency.",
       "the paragraph does not say", "abstain"),
]
print(f"gold set: {len(gold_set)} examples")

## 3. Score function and formatters

The score function is the same simple substring check. The helpers `per_bucket`, `overall`, and `fmt` print the ablation table cleanly.


In [ ]:
def score_fn(output: Answer, example: GoldExample[str, Answer]) -> float:
    if example.expected is None:
        return 0.0
    return 1.0 if example.expected.answer.lower().strip() in output.answer.lower().strip() else 0.0


def per_bucket(bundles, examples):
    by: dict[str, list[float]] = {}
    for b, e in zip(bundles, examples):
        by.setdefault(e.id, []).append(b.accuracy)
    return {k: sum(v) / len(v) for k, v in by.items()}


def overall(buckets):
    return sum(buckets.values()) / max(1, len(buckets))


def fmt(label, buckets):
    line = "  ".join(f"{k}={buckets.get(k, 0):.2f}" for k in ("factual", "inferential", "abstain"))
    return f"{label:<28} {line}  overall={overall(buckets):.2f}"


## 4. Baseline — seed topology, seed prompts

Row 1 of the ablation. Single CoT agent with the seed prompt.


In [ ]:
from orqest.optimization import TopologyEvaluator
from orqest.orchestration.spec import AgentStepSpec, PipelineSpec, PipelineStepEntry
from orqest.orchestration.hydrate import CallableRegistry

callable_registry = CallableRegistry()
seed_topology = PipelineSpec(steps=[PipelineStepEntry(operation=AgentStepSpec(agent_name="cot"))])

baseline_evaluator = TopologyEvaluator(
    score_fn=score_fn,
    callable_registry=callable_registry,
    agent_registry=seed_registry,
)
baseline_bundles = await baseline_evaluator.evaluate_batch({"main": seed_topology}, gold_set)
baseline_buckets = per_bucket(baseline_bundles, gold_set)
print(fmt("BASELINE (seed/seed)", baseline_buckets))

## 5. Phase 1 — topology search

Find a better topology. The prompts stay fixed at their seed values during this search.

We use a smaller budget (3 generations) than notebook 08 — this notebook stacks two phases, and we want to keep total cost reasonable.


In [ ]:
from orqest.optimization import MetaAgentConfig, MetaAgentSearch, TopologyGene

gene = TopologyGene(
    name="main",
    initial=seed_topology,
    constraints="Prefer simpler topologies. Add a verifier step only when it helps the inferential bucket.",
)

search = MetaAgentSearch(
    MetaAgentConfig(
        n_generations=3,
        archive_strategy="top_k",
        archive_size=2,
        reflexion_passes=1,
        debug_max=2,
        minibatch_size=3,
        seed=42,
    ),
    gene=gene,
    evaluator=baseline_evaluator,
    meta_agent_model=config.llm_model,
    api_key=config.llm_api_key,
)

topology_result = await search.run(trainset=gold_set)
winning_topology = topology_result.best_decoded["main"]

print(f"phase 1 best score (minibatch): {topology_result.best_score:.3f}")
print(f"winning topology kind:          {winning_topology.kind}")

## 6. Topology-only ablation row

Re-evaluate the **winning topology** with the **seed prompts** unchanged. This isolates the lift that came purely from topology search.


In [ ]:
topology_only_bundles = await baseline_evaluator.evaluate_batch(
    {"main": winning_topology}, gold_set,
)
topology_only_buckets = per_bucket(topology_only_bundles, gold_set)
print(fmt("TOPOLOGY only (evolved/seed)", topology_only_buckets))

## 7. Phase 2 — GEPA prompt evolution on the winning topology

Now we evolve the prompts, with the topology held fixed.

The first step is to find every prompt slot in the winning topology. A `Pipeline` may have many agents nested at different depths (inside a `Parallel`, inside a `RefinementLoop`, inside a `Router`). The helper `collect_prompt_slots` walks the spec tree recursively and gathers every `AgentStepSpec` it finds.

We then create one `PromptGene` per unique agent name. The genome is what GEPA mutates.


In [ ]:
from orqest.optimization import Evaluator, Genome, OptimizationConfig, OptimizationRunner, PromptGene
from orqest.orchestration.hydrate import topology_from_spec


def collect_prompt_slots(spec, prefix=""):
    """Walk a TopologySpec and collect every (slot_name, agent_name) it contains."""
    from orqest.orchestration.spec import (
        AgentStepSpec, FunctionStepSpec, PipelineSpec, ParallelSpec,
        RouterSpec, RefinementLoopSpec,
    )
    out: list[tuple[str, str]] = []
    if isinstance(spec, AgentStepSpec) and spec.agent_name:
        out.append((f"{prefix}{spec.agent_name}.system_prompt", spec.agent_name))
    elif isinstance(spec, PipelineSpec):
        for i, e in enumerate(spec.steps):
            out.extend(collect_prompt_slots(e.operation, f"{prefix}step{i}."))
    elif isinstance(spec, ParallelSpec):
        for i, s in enumerate(spec.steps):
            out.extend(collect_prompt_slots(s, f"{prefix}p{i}."))
    elif isinstance(spec, RouterSpec):
        for r in spec.routes:
            out.extend(collect_prompt_slots(r.step, f"{prefix}{r.name}."))
        if spec.fallback_step is not None:
            out.extend(collect_prompt_slots(spec.fallback_step, f"{prefix}fallback."))
    elif isinstance(spec, RefinementLoopSpec):
        out.extend(collect_prompt_slots(spec.step, f"{prefix}loop."))
    return out


SEED_PROMPTS = {"cot": SEED_COT_PROMPT, "verifier": SEED_VERIFIER_PROMPT}

slots = collect_prompt_slots(winning_topology)
unique_agent_names = sorted({a for _, a in slots})
print(f"unique agents in winning topology: {unique_agent_names}")

# One PromptGene per unique agent (multiple structural slots per agent share a prompt).
prompt_genes = [
    PromptGene(
        name=f"{a}.system_prompt",
        initial=SEED_PROMPTS.get(a, SEED_COT_PROMPT),
        constraints="Must end with 'the paragraph does not say' instruction for off-topic / abstain inputs.",
    )
    for a in unique_agent_names
]
genome = Genome(genes=prompt_genes)
print(f"GEPA genome: {len(genome.genes)} prompt gene(s)")

### 7b. The GEPA bridge

GEPA's `Evaluator` works on a single agent. We need to evaluate the **whole topology** with the candidate prompts. The trick: write a thin wrapper that pretends to be a `BaseAgent` (it has a `call_model` method), but internally hydrates the whole winning topology with prompts from the decoded genome.

This is one of those places where orqest's "everything is a Pydantic spec" design pays off: the topology can be re-hydrated with different prompts each time GEPA proposes a candidate, without any structural changes.


In [ ]:
def prompt_evolved_factory_for_agent(agent_name: str, decoded_prompts: dict[str, Any]):
    prompt = decoded_prompts.get(f"{agent_name}.system_prompt", SEED_PROMPTS.get(agent_name, SEED_COT_PROMPT))

    def _f():
        return CotAgent(
            agent_name=agent_name,
            system_prompt=prompt,
            output_type=Answer,
            model=config.llm_model,
            api_key=config.llm_api_key,
        )
    return _f


class _EvolvedTopologyAgent:
    """A thin wrapper that looks like a BaseAgent to GEPA's Evaluator.

    Internally it hydrates the winning topology with the candidate prompts and runs it.
    """

    def __init__(self, decoded_prompts):
        self._decoded = decoded_prompts

    async def call_model(self, prompt, state):
        agents = {a: prompt_evolved_factory_for_agent(a, self._decoded) for a in unique_agent_names}
        topology = topology_from_spec(
            winning_topology,
            callable_registry=callable_registry,
            agent_registry=agents,
        )
        run_result = await topology.run(prompt)

        # Unpack composite results (ParallelResult / LoopResult) into a single output.
        from orqest.orchestration.parallel import ParallelResult
        from orqest.orchestration.loop import LoopResult
        if isinstance(run_result, ParallelResult):
            inner = run_result.merged
            output = inner if not isinstance(inner, list) else (inner[0] if inner else None)
        elif isinstance(run_result, LoopResult):
            output = run_result.output
        else:
            output = getattr(run_result, "output", run_result)

        # Match GEPA's expected AgentRunResult-like shape.
        class _R: pass
        r = _R()
        r.output = output
        r.usage = lambda: None
        return r


def gepa_agent_factory(decoded):
    return _EvolvedTopologyAgent(decoded)


prompt_evaluator = Evaluator(agent_factory=gepa_agent_factory, score_fn=score_fn)
prompt_runner = OptimizationRunner(
    OptimizationConfig(
        max_metric_calls=20,
        reflection_model=config.llm_model,
        minibatch_size=3,
        valset_fraction=0.5,
        seed=42,
    ),
    genome=genome,
    evaluator=prompt_evaluator,
    api_key=config.llm_api_key,
)

prompt_result = await prompt_runner.optimize(trainset=gold_set)
print(f"phase 2 best score: {prompt_result.best_score:.3f}")

## 8. The full ablation

Now build the remaining two rows:

- **Prompts only** — seed topology, GEPA-evolved prompts.
- **Combined** — winning topology, GEPA-evolved prompts.

Then print the full table.


In [ ]:
# Prompts-only row: seed topology with GEPA-evolved prompts.
prompts_only_registry = {
    a: prompt_evolved_factory_for_agent(a, prompt_result.best_decoded)
    for a in unique_agent_names if a in seed_registry
}
prompts_only_evaluator = TopologyEvaluator(
    score_fn=score_fn,
    callable_registry=callable_registry,
    agent_registry=prompts_only_registry,
)
prompts_only_bundles = await prompts_only_evaluator.evaluate_batch({"main": seed_topology}, gold_set)
prompts_only_buckets = per_bucket(prompts_only_bundles, gold_set)

# Combined row: winning topology with GEPA-evolved prompts.
combined_registry = {
    a: prompt_evolved_factory_for_agent(a, prompt_result.best_decoded)
    for a in unique_agent_names
}
combined_evaluator = TopologyEvaluator(
    score_fn=score_fn,
    callable_registry=callable_registry,
    agent_registry=combined_registry,
)
combined_bundles = await combined_evaluator.evaluate_batch({"main": winning_topology}, gold_set)
combined_buckets = per_bucket(combined_bundles, gold_set)

print("=== ABLATION ===\n")
print(fmt("BASELINE      (seed/seed)",       baseline_buckets))
print(fmt("TOPOLOGY only (evolved/seed)",    topology_only_buckets))
print(fmt("PROMPTS only  (seed/evolved)",    prompts_only_buckets))
print(fmt("COMBINED      (evolved/evolved)", combined_buckets))

print()
print("Lift over baseline (overall):")
for label, buckets in [
    ("topology only", topology_only_buckets),
    ("prompts only",  prompts_only_buckets),
    ("combined",      combined_buckets),
]:
    print(f"  {label:<14} {overall(buckets) - overall(baseline_buckets):+.3f}")

## Reading the ablation honestly

A few things to look out for in your run:

**If all four rows tie at 1.00** — the baseline is already saturated. The notebook *demonstrates the wiring* but has nothing to claim about lift. This is actually common with strong models on small gold sets. Don't run a search against a benchmark your seed already aces.

**If COMBINED scores LOWER than baseline** — that is a real failure case, not a bug. It can happen with small gold sets: Phase 2 GEPA was tuned to the Phase 1 winning topology on a tiny minibatch. With limited examples, GEPA can converge on prompts that overfit to noise — and overfit prompts can hurt buckets they were not sampled from.

The mitigation is what a real production search would already do:

- Bigger gold set (or better, a held-out validation set).
- More reflexion passes.
- Larger minibatch per round.
- A weaker task model, so there is real headroom to climb.

This is the classic "search overfits to tiny eval set" failure. The right takeaway: **two-phase composition only beats either phase alone when there is real headroom to find**.


## Recap

The full pattern:

1. **Phase 1** — search over topologies with prompts held fixed. Cheap, structural.
2. **Phase 2** — search over prompts with topology held fixed. The bridge wrapper lets GEPA's single-agent `Evaluator` evaluate the whole hydrated topology.
3. **Ablation table** — measure each row so you can attribute lift to the right source.

**Why decouple?** Because nested search is `O(N × M)`. Two-phase is `O(N + M)`. That is what makes this approach affordable.

**What's next:**

- **MCTS over `TopologySpec`** — AFlow-style search using the same archive and metric machinery.
- **Per-step cost capture** — today `cost_usd=0.0` because per-step `Usage` isn't aggregated. A small refinement.
- **Sandboxed code generation** — eventually the meta agent could emit raw Python `forward()` for tasks where typed specs are too constrained.

See `docs/concepts/topology_optimization.md` for design rationale.
